# Hybrid Recommendation System

This notebook implements a hybrid recommendation system using **The Movies Dataset** (Kaggle - Rounak Banik) that combines:
- Popularity-based recommendations
- Content-based recommendations
- Weighted ensemble of multiple approaches
- Evaluate hybrid system performance

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from src.models import (PopularityRecommender, GenreBasedRecommender,
                        TFIDFContentRecommender, HybridRecommender)
from src.models.collaborative_filtering import UserBasedCF, ItemBasedCF, MatrixFactorizationCF
from src.evaluation import MetricsCalculator
from src.data import DataLoader

print("Modules loaded successfully!")

Modules loaded successfully!


# Import data preprocessor
from src.data.data_preprocessor import DataPreprocessor
preprocessor = DataPreprocessor()

# Load data with user-based split to ensure users appear in both train and test
train_ratings, test_ratings = preprocessor.create_train_test_split(ratings, test_size=0.2, method='user_based')

# Filter for overlapping users between train and test
train_users = set(train_ratings['userId'].unique())
test_users = set(test_ratings['userId'].unique())
overlap_users = train_users & test_users
print(f"Train users: {len(train_users)}, Test users: {len(test_users)}, Overlap: {len(overlap_users)}")

# Keep only overlapping users
train_ratings = train_ratings[train_ratings['userId'].isin(overlap_users)]
test_ratings = test_ratings[test_ratings['userId'].isin(overlap_users)]

# Sample for faster training
train_ratings = train_ratings.sample(n=100000, random_state=42)
test_ratings = test_ratings.sample(n=20000, random_state=42)

print(f"Train ratings: {len(train_ratings)}")
print(f"Test ratings: {len(test_ratings)}")
print(f"Unique users in train: {train_ratings['userId'].nunique()}")
print(f"Unique movies in train: {train_ratings['movieId'].nunique()}")

In [2]:
# FIX #1 — Load from correctly split preprocessed files
loader = DataLoader()
movies_metadata, train_ratings, test_ratings = loader.load_preprocessed_data()

# Enforce overlap
overlap_users = set(train_ratings['userId']) & set(test_ratings['userId'])
train_ratings = train_ratings[train_ratings['userId'].isin(overlap_users)].copy()
test_ratings  = test_ratings[test_ratings['userId'].isin(overlap_users)].copy()

print(f"Overlap users:  {len(overlap_users):,}")
print(f"Train ratings:  {len(train_ratings):,}")
print(f"Test ratings:   {len(test_ratings):,}")

assert len(overlap_users) >= 95, f"Too few overlap users: {len(overlap_users)}"
print("✓ Verification gate: overlap users ≥ 95")

# ── Prepare CF training subset ───────────────────────────────────────────────
# Pre-select evaluation users so cf_train stays ~1.2M rows (not 20M)
rng = np.random.default_rng(42)

eval_users = MetricsCalculator._select_evaluation_users(
    test_ratings=test_ratings,
    train_ratings=train_ratings,
    n_users=100,
    random_state=42,
)
eval_user_set = set(int(u) for u in eval_users)

eval_user_train = train_ratings[train_ratings['userId'].isin(eval_user_set)]

other_train = train_ratings[~train_ratings['userId'].isin(eval_user_set)]
other_counts = other_train.groupby('userId').size()
other_eligible = other_counts[other_counts >= 3].index.tolist()
n_extra = min(14_900, len(other_eligible))
extra_users = set(rng.choice(other_eligible, size=n_extra, replace=False).tolist())

cf_train = pd.concat([
    eval_user_train,
    other_train[other_train['userId'].isin(extra_users)],
], ignore_index=True)

# Free ~1.34 GB — other_train is a 99.96% copy of train_ratings and must be
# released before the user_cf similarity matrix (1.8 GB) is computed.
del eval_user_train, other_train, other_counts, other_eligible
import gc; gc.collect()

print(f"\nCF train: {len(cf_train):,} ratings, "
      f"{cf_train['userId'].nunique():,} users, "
      f"{cf_train['movieId'].nunique():,} movies")
assert len(cf_train) >= 15_000
print("✓ Verification gate: CF ratings ≥ 15,000")

Overlap users:  256,107
Train ratings:  20,881,168
Test ratings:   5,109,682
✓ Verification gate: overlap users ≥ 95



CF train: 1,198,121 ratings, 15,000 users, 21,650 movies
✓ Verification gate: CF ratings ≥ 15,000


## Initialize Base Models

In [3]:
# Train all base models
print("Training popularity baseline...")
pop_recommender = PopularityRecommender(train_ratings, movies_metadata)

print("Training genre content model...")
genre_recommender = GenreBasedRecommender(movies_metadata, train_ratings, use_keywords=True)

print("Training TF-IDF content model...")
tfidf_recommender = TFIDFContentRecommender(
    movies_metadata, train_ratings,
    max_features=10000, min_df=2, max_df=0.8, min_user_rating=4.0,
)

# Centered SVD-MF on FULL train_ratings (20.9M rows, 256k users, n_components=50).
# n=50 selected from CF notebook grid search (best val_RMSE=0.8601).
# validate=False skips the grid-search to avoid OOM (3 copies of a 1.3 GB DataFrame).
# _build_model() frees filtered (~1.3 GB) before the SVD fit when validate=False.
print("Training centered SVD MF on FULL train_ratings (n_components=50)...")
mf_cf = MatrixFactorizationCF(
    train_ratings, n_components=50, min_ratings=3,
    max_users=None, max_movies=None, random_state=42, validate=False,
)

# Hybrid: SVD-MF is the sole collaborative signal (collab_weight=0.95).
# Why single signal: pooling item_cf (P@10=0.035) with SVD via mean(icf_norm, mf_norm)
# creates score ties between SVD rank-1 (correct) and item_cf rank-1 (wrong), halving
# effective quality. TF-IDF (weight=0.05) adds content diversity without contaminating
# top-10 since TF-IDF-only candidates max at 0.05 while SVD top-10 scores are ≥0.855.
print("Building hybrid recommender (SVD-only collab, content_weight=0.05)...")
hybrid_recommender = HybridRecommender(
    pop_recommender=pop_recommender,
    genre_recommender=genre_recommender,
    movies_df=movies_metadata,
    train_ratings=train_ratings,
    user_cf_recommender=None,     # excluded: 1.8 GB user_similarity matrix OOMs
    item_cf_recommender=None,     # excluded: mean-pooling with SVD kills quality signal
    mf_recommender=mf_cf,
    content_recommender=tfidf_recommender,
    content_weight=0.05,
    collab_weight=0.95,
    candidate_pool_size=500,
    cold_start_threshold=3,
    cold_start_pop_weight=0.30,
)

test_user = list(eval_user_set)[0]
sample_recs = hybrid_recommender.recommend(test_user, n=10)
print(f"\nSample hybrid recs for user {test_user}:")
print(sample_recs[['movieId', 'hybrid_score']].to_string(index=False))

Training popularity baseline...


Training genre content model...


Genre-based model: 20 genres, 19387 keywords, user prefs computed on-demand
Training TF-IDF content model...


TF-IDF content model: 42210 movies built
Training centered SVD MF on FULL train_ratings (n_components=50)...


Matrix Factorization (centered SVD n=50): 256107 users, 30446 movies, 20863370 ratings
Building hybrid recommender (SVD-only collab, content_weight=0.05)...



Sample hybrid recs for user 18432:
 movieId  hybrid_score
     924      0.950000
    2028      0.865996
    1206      0.605509
    1219      0.595215
    1252      0.590933
    1201      0.470162
    1228      0.466898
     904      0.466527
    1207      0.450706
     364      0.423319


## Hybrid Recommendation System

In [4]:
# Evaluate hybrid — SAME settings as all other notebooks
calculator = MetricsCalculator()
eval_kwargs = dict(k=10, n_users=100, min_rating=4.0, random_state=42,
                   train_ratings=train_ratings)

print("Evaluating Hybrid Recommender (two-tier, FIX #4)...")
hybrid_metrics = calculator.evaluate_model_with_recommendations(
    hybrid_recommender, test_ratings, **eval_kwargs)

print("\nHybrid Recommender Metrics:")
for k, v in hybrid_metrics.items():
    print(f"  {k}: {v:.4f}")

# VERIFICATION GATE #4: coverage ≥ 98%
cov = hybrid_metrics['coverage']
print(f"\nHybrid coverage: {cov:.3f}")
if cov < 0.98:
    print(f"WARNING: coverage {cov:.3f} < 0.98 — check that CF users overlap test set")
else:
    print("✓ Verification gate: hybrid coverage ≥ 98%")

Evaluating Hybrid Recommender (two-tier, FIX #4)...



Hybrid Recommender Metrics:
  precision@k: 0.1024
  recall@k: 0.0714
  f1@k: 0.0841
  hit_rate: 0.0860
  evaluated_users: 100.0000
  coverage: 1.0000
  catalog_coverage: 0.0068

Hybrid coverage: 1.000
✓ Verification gate: hybrid coverage ≥ 98%


## Evaluate Hybrid System

In [5]:
# Save hybrid metrics
calculator.save_metrics(hybrid_metrics, '../reports/hybrid_metrics.json')
print("Hybrid metrics saved.")

# ── Full comparison across all models ────────────────────────────────────────
print("\n" + "="*65)
print("FULL MODEL COMPARISON (K=10, threshold=4.0, same 100 test users)")
print("="*65)

pop_metrics   = calculator.evaluate_model_with_recommendations(pop_recommender,   test_ratings, **eval_kwargs)
genre_metrics = calculator.evaluate_model_with_recommendations(genre_recommender, test_ratings, **eval_kwargs)
tfidf_metrics = calculator.evaluate_model_with_recommendations(tfidf_recommender, test_ratings, **eval_kwargs)
mf_metrics    = calculator.evaluate_model_with_recommendations(mf_cf,             test_ratings, **eval_kwargs)

all_results = {
    'Popularity':     pop_metrics,
    'Genre CB':       genre_metrics,
    'TF-IDF CB':      tfidf_metrics,
    'SVD-MF (full)':  mf_metrics,
    'Hybrid':         hybrid_metrics,
}

print(f"\n{'Model':22s} {'P@10':>7} {'R@10':>7} {'F1@10':>7} {'HitRate':>8} {'Coverage':>9} {'Users':>6}")
print("-"*67)
for model, m in all_results.items():
    print(f"{model:22s} {m['precision@k']:7.4f} {m['recall@k']:7.4f} "
          f"{m['f1@k']:7.4f} {m['hit_rate']:8.4f} {m['coverage']:9.3f} {m['evaluated_users']:6.0f}")

all_models_metrics = {
    'popularity':           pop_metrics,
    'matrix_factorization': mf_metrics,
    'genre_content':        genre_metrics,
    'tfidf_content':        tfidf_metrics,
    'hybrid':               hybrid_metrics,
}
calculator.save_metrics(all_models_metrics, '../reports/all_models_metrics.json')
print("\n✓ All model metrics saved → reports/all_models_metrics.json")

hybrid_p = hybrid_metrics['precision@k']
pop_p    = pop_metrics['precision@k']
mf_p     = mf_metrics['precision@k']
print(f"\nSVD-MF (full) P@10: {mf_p:.4f}  |  Hybrid P@10: {hybrid_p:.4f}")
if hybrid_p > 0.20:
    print(f"✓ TIER 2 achieved: Hybrid P@10 {hybrid_p:.4f} > 0.20")
elif hybrid_p > 0.15:
    print(f"✓ TIER 1 achieved: Hybrid P@10 {hybrid_p:.4f} > 0.15")
elif hybrid_p > pop_p:
    print(f"TIER 3 achieved: Hybrid {hybrid_p:.4f} > Popularity {pop_p:.4f}")
else:
    print(f"⚠ Below Tier 1: Hybrid P@10 {hybrid_p:.4f}")

Hybrid metrics saved.

FULL MODEL COMPARISON (K=10, threshold=4.0, same 100 test users)



Model                     P@10    R@10   F1@10  HitRate  Coverage  Users
-------------------------------------------------------------------
Popularity              0.0429  0.0347  0.0383   0.0360     1.000    100
Genre CB                0.0107  0.0066  0.0082   0.0090     1.000    100
TF-IDF CB               0.0071  0.0073  0.0072   0.0060     1.000    100
SVD-MF (full)           0.1036  0.0722  0.0851   0.0870     1.000    100
Hybrid                  0.1024  0.0714  0.0841   0.0860     1.000    100

✓ All model metrics saved → reports/all_models_metrics.json

SVD-MF (full) P@10: 0.1036  |  Hybrid P@10: 0.1024
TIER 3 achieved: Hybrid 0.1024 > Popularity 0.0429


## Save Hybrid Model

**DISABLED — redundant summary (already printed in cell-9)**

## Model Comparison Summary

**DISABLED (stale cell — re-evaluates with n_users=50 and no min_rating, overwriting consistent metrics from cell-9)**

The final comparison table in cell-9 uses the correct contract: K=10, min_rating=4.0, n_users=100, random_state=42.